# Data Pre-Processing

### Import Library

In [196]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Memuat Dataset

In [197]:
df = pd.read_csv('Assignment-2_Data.csv')
df.columns = df.columns.str.strip()
print('Pemuatan Dataset Berhasil')

Pemuatan Dataset Berhasil


C:\Users\Hype 5 AMD X3\AppData\Local\Temp\ipykernel_19684\658044321.py:1: DtypeWarning: Columns (1:  age, 6:  balance) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('Assignment-2_Data.csv')


### Melihat Sampel Data

In [198]:
df.head()

,Id,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,1001,999,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,1002,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,1003,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,1004,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,1005,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


### Melihat Dimensi Data Sebelum Penghapusan Baris dan Kolom

In [199]:
df.shape

(45211, 18)

### Melihat Missing Value pada Dataset

In [200]:
df.isnull().sum()

Id           0
age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

### Melihat Duplikat Data pada Dataset

In [201]:
df.duplicated().sum()

np.int64(0)

### Menghapus Kolom Id

In [202]:
df.drop(columns=['Id'], inplace=True)

### Menghapus Missing Value

In [203]:
df.dropna(inplace=True)
df.reset_index(drop=True, inplace=True)

### Melihat Missing Value Setelah Dihapus

In [204]:
df.isnull().sum()

age          0
job          0
marital      0
education    0
default      0
balance      0
housing      0
loan         0
contact      0
day          0
month        0
duration     0
campaign     0
pdays        0
previous     0
poutcome     0
y            0
dtype: int64

### Melihat Dimensi Data Setelah Penghapusan

In [205]:
df.shape

(45211, 17)

### Melihat Usia yang Anomali (Usia Normal = 18-100) -> Mengubah Usia yang Anomali dengan Imputasi Median

In [206]:
df['age'] = pd.to_numeric(df['age'], errors='coerce')
df['balance'] = pd.to_numeric(df['balance'], errors='coerce')

In [207]:
kondisi = (df['age'] < 18) | (df['age'] > 100)
baris = df[kondisi]

print(f"Total baris anomali: {len(baris)}")
print(baris['age'])

Total baris anomali: 4
0        999.0
45203     -1.0
45204     -1.0
45205     -1.0
Name: age, dtype: float64


In [208]:
median_umur = df.loc[(df['age'] >= 18) & (df['age'] <= 100), 'age'].median()
df.loc[(df['age'] < 18) | (df['age'] > 100), 'age'] = median_umur

sisa = df[(df['age'] < 18) | (df['age'] > 100)]
print(f"Jumlah anomali umur: {len(sisa)}")

Jumlah anomali umur: 0


### Melihat Jumlah Unknown pada Kolom Job dan Education -> Mengubah Unknown dengan Imputasi Modus

In [209]:
print("Jumlah 'unknown' pada job:", (df['job'] == 'unknown').sum())
print("Jumlah 'unknown' pada education:", (df['education'] == 'unknown').sum())

Jumlah 'unknown' pada job: 0
Jumlah 'unknown' pada education: 0


In [210]:
for col in ['job', 'education']:
    modus = df[df[col] != 'unknown'][col].mode()[0]
    df[col] = df[col].replace('unknown', modus)

In [211]:
print("Jumlah 'unknown' pada job:", (df['job'] == 'unknown').sum())
print("Jumlah 'unknown' pada education:", (df['education'] == 'unknown').sum())

Jumlah 'unknown' pada job: 0
Jumlah 'unknown' pada education: 0


### Melihat Jumlah Outlier pada Kolom Age, Balance, Duration, dan Campaign -> Membatasi Jumlah Outlier dengan Minimum dan Maksimum Tertentu

In [212]:
fitur_numerik = ['age', 'balance', 'duration', 'campaign']

for col in fitur_numerik:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR
    
    outlier_awal = ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()
    
    df[col] = np.clip(df[col], batas_bawah, batas_atas)
    
    outlier_akhir = ((df[col] < batas_bawah) | (df[col] > batas_atas)).sum()
    
    print(f"{col:10} | Outlier Awal: {outlier_awal:4} -> Sisa: {outlier_akhir}")

age        | Outlier Awal:  486 -> Sisa: 0
balance    | Outlier Awal: 4729 -> Sisa: 0
duration   | Outlier Awal: 3235 -> Sisa: 0
campaign   | Outlier Awal: 3064 -> Sisa: 0


### Pemisahan Fitur (X) dan Target (y)

In [213]:
X = df.drop(columns=['y'])
y = df['y'].map({'no': 0, 'yes': 1})

X = pd.get_dummies(X, drop_first=True, dtype=int)

### Pembagian Data Latih dan Data Uji

In [214]:
np.random.seed(43)
indeks_acak = np.random.permutation(len(X))
batas_latih = int(0.8 * len(X))

X_train = X.iloc[indeks_acak[:batas_latih]].copy()
X_test  = X.iloc[indeks_acak[batas_latih:]].copy()
y_train = y.iloc[indeks_acak[:batas_latih]].copy()
y_test  = y.iloc[indeks_acak[batas_latih:]].copy()

print(f"Data Latih (X_train): {X_train.shape[0]} baris")
print(f"Data Uji   (X_test) : {X_test.shape[0]} baris")

Data Latih (X_train): 36168 baris
Data Uji   (X_test) : 9043 baris


### Manual Standardization

In [215]:
rata_rata = X_train.mean()
standar_deviasi = X_train.std()

standar_deviasi[standar_deviasi == 0] = 1.0

X_train_scaled = (X_train - rata_rata) / standar_deviasi
X_test_scaled  = (X_test - rata_rata) / standar_deviasi

---

# Modelling (Bagian Awal)

### Menetukan Nilai K Secara Langsung

In [216]:
k = 5
print(f"Nilai K adalah: {k}")

Nilai K adalah: 5


### Mengukur Jarak dengan Menggunakan Metrik Euclidean Distance

In [217]:
def hitung_euclidean(nasabah_baru, X_latih):
    jarak = np.sqrt(np.sum((X_latih - nasabah_baru) ** 2, axis=1))
    return jarak

In [218]:
data_latih_np = X_train_scaled.values
nasabah_uji_1 = X_test_scaled.iloc[0].values

hasil_jarak = hitung_euclidean(nasabah_uji_1, data_latih_np)

print("Total jarak yang berhasil dihitung :", len(hasil_jarak))
print(f"{k} Jarak nasabah yang paling dekat :", np.sort(hasil_jarak)[:k])

Total jarak yang berhasil dihitung : 36168
5 Jarak nasabah yang paling dekat : [1.29931086 1.32051094 1.49546084 1.66952066 2.08186272]


In [219]:
indeks = np.argsort(hasil_jarak)[:k]

target_latih = y_train.values
label  = target_latih[indeks]

hasil = 1 if np.mean(label) >= 0.5 else 0

print(f"Indeks {k} nasabah terdekat di data latih     : {indeks}")
print(f"Label asli dari {k} tetangga tersebut         : {label}")
print(f"Hasil tebakan model untuk nasabah uji ke-1  : {hasil} ({'yes' if hasil == 1 else 'no'})")
print(f"Keputusan asli yang sebenarnya              : {y_test.iloc[0]} ({'yes' if y_test.iloc[0] == 1 else 'no'})")

Indeks 5 nasabah terdekat di data latih     : [30044 33371  8259 31636 16825]
Label asli dari 5 tetangga tersebut         : [nan nan nan nan nan]
Hasil tebakan model untuk nasabah uji ke-1  : 0 (no)
Keputusan asli yang sebenarnya              : nan (no)
